# 04 - Statistical Analysis

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../data/processed/air_quality_cleaned.csv")

pm25 = df[df["pollutant_id"] == "PM2.5"]["pollutant_avg"].dropna()
pm10 = df[df["pollutant_id"] == "PM10"]["pollutant_avg"].dropna()

print("PM2.5 count:", len(pm25))
print("PM10  count:", len(pm10))

PM2.5 count: 447
PM10  count: 444


In [2]:
# mean, median, std for PM2.5 and PM10
print("PM2.5 - mean:", round(pm25.mean(), 2), "median:", round(pm25.median(), 2), "std:", round(pm25.std(), 2))
print("PM10  - mean:", round(pm10.mean(),  2), "median:", round(pm10.median(),  2), "std:", round(pm10.std(),  2))

PM2.5 - mean: 175.23 median: 149.0 std: 105.17
PM10  - mean: 151.78 median: 132.5 std: 79.32


In [3]:
# skewness - how spread out / one-sided the data is
print("PM2.5 skewness:", round(pm25.skew(), 2))
print("PM10  skewness:", round(pm10.skew(),  2))

PM2.5 skewness: 0.56
PM10  skewness: 1.15


In [4]:
# Pearson correlation between PM2.5 and PM10
# only stations that have both readings
pm25_df = df[df["pollutant_id"] == "PM2.5"][["station", "pollutant_avg"]].rename(columns={"pollutant_avg": "PM2.5"})
pm10_df = df[df["pollutant_id"] == "PM10"][["station",  "pollutant_avg"]].rename(columns={"pollutant_avg": "PM10"})

merged = pm25_df.merge(pm10_df, on="station")
print("Stations with both readings:", len(merged))

r, p = stats.pearsonr(merged["PM2.5"], merged["PM10"])
print("Pearson r:", round(r, 3), "  p-value:", round(p, 4))

Stations with both readings: 431
Pearson r: 0.881   p-value: 0.0


In [5]:
# Kruskal-Wallis test - do PM2.5 levels differ across states?
groups = []
for state, group in df[df["pollutant_id"] == "PM2.5"].groupby("state"):
    if len(group) >= 3:
        groups.append(group["pollutant_avg"].values)

print("States used:", len(groups))
h, p = stats.kruskal(*groups)
print("H statistic:", round(h, 2), "  p-value:", round(p, 4))

States used: 20
H statistic: 252.22   p-value: 0.0


In [6]:
# most vs least polluted state - Mann-Whitney U test
state_means = df[df["pollutant_id"] == "PM2.5"].groupby("state")["pollutant_avg"].mean().sort_values()

least = state_means.index[0]
most  = state_means.index[-1]

x = df[(df["pollutant_id"] == "PM2.5") & (df["state"] == most)]["pollutant_avg"]
y = df[(df["pollutant_id"] == "PM2.5") & (df["state"] == least)]["pollutant_avg"]

u, p = stats.mannwhitneyu(x, y, alternative="greater")
print("Most polluted state :", most,  "mean PM2.5 =", round(x.mean(), 1))
print("Least polluted state:", least, "mean PM2.5 =", round(y.mean(), 1))
print("U:", round(u, 1), "  p-value:", round(p, 4))

Most polluted state : Delhi mean PM2.5 = 358.5
Least polluted state: Mizoram mean PM2.5 = 46.0
U: 37.0   p-value: 0.0503


In [7]:
# summary stats per pollutant
summary = df.groupby("pollutant_id")["pollutant_avg"].agg(["mean", "median", "std", "min", "max", "count"]).round(2)
summary.to_csv("../outputs/tables/pollutant_stats.csv")
summary

,mean,median,std,min,max,count
pollutant_id,,,,,,
CO,51.78,47.0,27.48,1.0,127.0,459
NH3,7.11,5.0,6.31,1.0,45.0,402
NO2,36.75,31.0,28.61,1.0,229.0,455
OZONE,18.67,13.0,21.78,1.0,302.0,450
PM10,151.78,132.5,79.32,23.0,436.0,444
PM2.5,175.23,149.0,105.17,15.0,410.0,447
SO2,17.75,14.0,13.95,1.0,113.0,436
